HuggingFace Trainer API — production NLP training
Big picture first

Day 2:

You manually trained BERT.

You wrote:

forward pass

loss

backward

optimizer step

scheduler

validation

saving

That taught:

how learning actually works.

Now:

HuggingFace Trainer does those repetitive engineering tasks for you.

Short BERT vs DistilBERT

 |          | BERT            | DistilBERT  |
| -------- | --------------- | ----------- |
| Layers   | 12              | 6           |
| Params   | 110M            | 66M         |
| Speed    | slower          | faster      |
| Size     | larger          | smaller     |
| Accuracy | slightly higher | nearly same |


| Feature                         | What it does                                                          | Why it matters                                              | Manual loop equivalent                                  |
| ------------------------------- | --------------------------------------------------------------------- | ----------------------------------------------------------- | ------------------------------------------------------- |
| **Gradient Accumulation**       | Accumulates gradients across multiple batches before updating weights | Simulates larger batch sizes when RAM/GPU memory is limited | Manually delay `optimizer.step()` after several batches |
| **Mixed Precision (FP16/BF16)** | Uses lower precision numbers for faster training and less memory      | Speeds up training significantly on supported GPUs          | `torch.cuda.amp.autocast()` + `GradScaler()`            |
| **Distributed Training**        | Splits training across multiple GPUs or machines                      | Enables large-model or faster training                      | `DistributedDataParallel (DDP)` setup                   |
| **Checkpointing**               | Saves model state automatically during training                       | Prevents losing progress and allows resume                  | `torch.save()` after epochs                             |
| **Logging**                     | Tracks loss, accuracy, learning rate, etc.                            | Helps monitor training and debug issues                     | `print()` statements or W&B logging                     |
| **Early Stopping**              | Stops training when validation performance stops improving            | Prevents overfitting and wasted compute                     | Manual val-check + break condition                      |
| **Evaluation**                  | Runs validation automatically at chosen intervals                     | Gives consistent performance monitoring                     | Manual `model.eval()` + validation loop                 |


In [ ]:
import sys
!{sys.executable} -m pip install evaluate

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)
from datasets import load_dataset
import evaluate
import numpy as np

MODEL_NAME = 'distilbert-base-uncased'

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model     = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=2
)

print(model.config.id2label)   # {0: 'LABEL_0', 1: 'LABEL_1'}
print("Params:", sum(p.numel() for p in model.parameters()))

Big Picture — what are we preparing?

Before training with Trainer, we need 4 things:

| Component             | Purpose                          |
| --------------------- | -------------------------------- |
| **Tokenizer**         | Convert text → tokens/numbers    |
| **Model**             | DistilBERT + classification head |
| **TrainingArguments** | Training rules/settings          |
| **Trainer**           | Runs the whole training process  |


Overall flow of this setup

Nothing is training yet.

You're just preparing:

Choose model
        ↓
Load tokenizer
        ↓
Load pretrained DistilBERT
        ↓
Attach sentiment classifier
        ↓
Prepare Trainer ecosystem
        ↓
Ready for training

In [ ]:
from datasets import load_dataset

dataset = load_dataset('glue', 'sst2')

def tokenize(batch):
    return tokenizer(batch['sentence'], truncation=True, max_length=128)

tokenized = dataset.map(tokenize, batched=True, remove_columns=['sentence','idx'])
tokenized = tokenized.rename_column('label', 'labels')
tokenized.set_format('torch')

# Use a subset for fast iteration on CPU
train_data = tokenized['train'].select(range(4000))
val_data   = tokenized['validation'].select(range(500))

print(train_data[0].keys())   # input_ids, attention_mask, labels

In [ ]:
import evaluate
import numpy as np

accuracy  = evaluate.load('accuracy')
f1_metric = evaluate.load('f1')

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    acc = accuracy.compute(predictions=preds, references=labels)['accuracy']
    f1  = f1_metric.compute(predictions=preds, references=labels,
                             average='binary')['f1']
    return {'accuracy': acc, 'f1': f1}

Final dataset pipeline

So overall:

SST-2 raw text

        ↓

Tokenize

        ↓

Remove unused columns

        ↓

Rename labels

        ↓

Convert to tensors

        ↓

Small CPU subset

        ↓

Trainer-ready dataset

Big picture

This section prepares both inputs and evaluation.

Pipeline:


Raw SST-2

      ↓

Tokenize + clean

      ↓

Trainer-ready tensors

      ↓

Trainer trains model

      ↓

compute_metrics()

      ↓

Accuracy + F1 reported automatically

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir='./distilbert-sst2',        # where to save checkpoints

    # Training schedule
    num_train_epochs=3,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,

    # Optimiser
    learning_rate=2e-5,
    weight_decay=0.01,                     # AdamW weight decay
    warmup_ratio=0.1,                      # 10% of steps for warmup

    # Evaluation
    eval_strategy='epoch',                 # evaluate after every epoch
    save_strategy='epoch',                 # save checkpoint every epoch
    load_best_model_at_end=True,           # reload best checkpoint when done
    metric_for_best_model='f1',            # use F1 to pick best, not loss

    # Logging
    logging_steps=50,
    report_to='none',                      # swap to 'wandb' if you want W&B
)

In [ ]:
from transformers import DataCollatorWithPadding, Trainer

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=val_data,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

# One line — replaces your entire Day 2 training loop
trainer.train()

Part 1 — TrainingArguments

This is the settings object for Trainer.

Without it:

Trainer exists

but has no instructions

With it:

Trainer knows:
how long to train,
when to evaluate,
when to save,
what metric matters

Think of it like game settings before starting a match.

Overall picture

TrainingArguments
        ↓

Training schedule

Optimiser settings

Evaluation rules

Checkpoint rules

Logging rules

Final mental model

Everything now looks like:

Raw text
     ↓
Tokenizer
     ↓
Dataset
     ↓
TrainingArguments
     ↓
Trainer
     ↓
trainer.train()
     ↓
Fine-tuned DistilBERT

Main takeaway

TrainingArguments = training rules

Trainer = training engine

trainer.train() = run the whole NLP pipeline automatically.

In [ ]:
# Full evaluation on val set
results = trainer.evaluate()
print(results)
# {'eval_loss': ..., 'eval_accuracy': ..., 'eval_f1': ...}

# Detailed predictions
predictions = trainer.predict(val_data)
logits      = predictions.predictions
labels      = predictions.label_ids
preds       = np.argmax(logits, axis=-1)

from sklearn.metrics import classification_report
print(classification_report(labels, preds, target_names=['negative','positive']))

Typical output:

{
 'eval_loss': 0.28,
 'eval_accuracy': 0.91,
 'eval_f1': 0.91
}

Meaning:

| Metric        | Meaning                       |
| ------------- | ----------------------------- |
| eval_loss     | How wrong predictions are     |
| eval_accuracy | % predictions correct         |
| eval_f1       | Balance of precision + recall |

Mental Flow of This Section

trainer.evaluate()
        ↓
overall metrics

trainer.predict()
        ↓
raw logits + labels

argmax()
        ↓
final predictions

classification_report()
        ↓
deep error analysis

So this section is basically:
"Training is done — now inspect whether the model deserves trust."

In [ ]:
# Save everything — model weights, tokenizer config, training args
trainer.save_model('./distilbert-sst2-final')
tokenizer.save_pretrained('./distilbert-sst2-final')

# The slickest way to use it after saving — HuggingFace pipeline
from transformers import pipeline

classifier = pipeline(
    'text-classification',
    model='./distilbert-sst2-final',
    tokenizer='./distilbert-sst2-final',
    device=-1   # -1 = CPU
)

test = [
    "One of the best films I've seen this decade.",
    "Complete garbage. Don't waste your time.",
    "It had its moments but overall disappointing.",
    "Surprisingly moving and beautifully shot.",
]

results = classifier(test)
for text, res in zip(test, results):
    print(f"[{res['label']:8s} {res['score']:.1%}]  {text}")